# Data Mining Assignment
## Apache Spark Data Lake Preprocessing in Google Colab


### 1. Install & initialize PySpark

In [23]:
# install PySpark
!pip -q install pyspark

In [24]:
# necessary imports
import os
import shutil
from google.colab import files
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

spark = SparkSession.builder.appName("SalesDataLakePreprocessing").getOrCreate()
print("Spark version:", spark.version)

Spark version: 4.0.5


#### 2. Load and Inspect with Spark
• Install PySpark, create a SparkSession, and read sales.csv from Bronze.

• Display the schema, a few rows, and the total record count.

• Use Spark operations to identify duplicates, null values, text-format inconsistencies, invalid numerical values, and inconsistent/invalid dates.


In [25]:
## read sales.csv from Bronze
# (header, "true") tells the Spark to use the first row of the csv as the column names rather than treating it as a row of data.
# (inferSchema, "true") direct the Spark to scan the file and guess the appropriate data types for each column.
# Without this option, Spark defaults every column to StringType
bronze_file = "/content/data_lake/bronze/sales.csv"
silver_path = "/content/data_lake/silver/preprocessed_sales/"
submission_csv = "/content/submissions/silver/sales_silver.csv"

df = spark.read.option("header", True).option("inferSchema", True).csv(bronze_file)

# print the column names, data types, and nullability of your DataFrame in a readable tree format
df.printSchema()

df.show(10, truncate=False)

# row count
before_count = df.count()
print("Number of rows: ", before_count)

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+---------------+---------------------+---------+--------+----------+--------

#### • Use Spark operations to identify duplicates, null values, text-format inconsistencies, invalid numerical values, and inconsistent/invalid dates.

In [26]:
# duplicate entries
total_count = df.count()
duplicate = total_count - df.dropDuplicates().count()
print("All duplicate rows: ", duplicate)

All duplicate rows:  10


In [27]:
# missing values
missings = []
for field in df.schema.fields:
    col_ = F.col(field.name)
    missing = col_.isNull()
    if isinstance(field.dataType, StringType):
        missing = missing | (F.trim(col_) == "")
    missings.append(F.sum(F.when(missing, 1).otherwise(0)).alias(field.name))

print("Missing values by column:")
df.select(*missings).show(truncate=False)

Missing values by column:
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|0       |0          |10           |0      |0       |0       |0         |0               |8             |8   |0    |0         |0           |
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



In [28]:
print("Distinct raw categories",)
df.groupBy("category").count().orderBy("category").show(50, truncate=False)

Distinct raw categories
+---------------+-----+
|category       |count|
+---------------+-----+
|Beauty         |136  |
|Books          |128  |
|CLOTHING       |1    |
|Clothing       |108  |
|ELECTRONICS    |1    |
|Electronics    |103  |
|Furniture      |132  |
|Grocery        |136  |
|Home Appliances|115  |
|SPORTS         |1    |
|Sports         |136  |
|beauty         |2    |
|sports         |1    |
+---------------+-----+



In [29]:
print("Distinct raw payment_method")
df.groupBy("payment_method").count().orderBy("payment_method").show(50, truncate=False)

Distinct raw payment_method
+----------------+-----+
|payment_method  |count|
+----------------+-----+
|NULL            |8    |
|Cash on Delivery|208  |
|Credit Card     |207  |
|Debit Card      |201  |
|Net Banking     |206  |
|UPI             |170  |
+----------------+-----+



In [30]:
print("Distinct raw cities")
df.groupBy("city").count().orderBy("city").show()

Distinct raw cities
+-----------+-----+
|       city|count|
+-----------+-----+
|       NULL|    8|
| Hyderabad |    2|
|    Mumbai |    1|
|  Bengaluru|   91|
|    Chennai|   88|
| Coimbatore|   81|
|      DELHI|    2|
|      Delhi|   81|
|  HYDERABAD|    1|
|  Hyderabad|   94|
|     JAIPUR|    1|
|     Jaipur|   84|
|    Kolkata|   81|
|    Lucknow|   90|
|     Mumbai|  106|
|       Pune|   82|
| Vijayawada|  105|
| coimbatore|    1|
|      delhi|    1|
+-----------+-----+



In [31]:
print("Distinct raw order_status")
df.groupBy("order_status").count().orderBy("order_status").show()

Distinct raw order_status
+------------+-----+
|order_status|count|
+------------+-----+
|   Cancelled|   77|
|   Delivered|  587|
|  Processing|  111|
|    Returned|   57|
|     Shipped|  168|
+------------+-----+



In [32]:
# Invalid and null numeric values
num_invalid = (
    F.col("quantity").isNull() | (F.col("quantity") <= 0) |
    F.col("unit_price").isNull() | (F.col("unit_price") <= 0) |
    F.col("discount_percent").isNull() | (F.col("discount_percent") < 0) | (F.col("discount_percent") > 100)
)

print("Rows with invalid numeric values:", df.filter(num_invalid).count())

# inspect invalid rows to see what went wrong
df.filter(num_invalid).show(truncate=False)


Rows with invalid numeric values: 28
+--------+-----------+----------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name   |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+----------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100034  |C0192      |Divya Sharma    |Yoga Mat              |Sports         |7       |1720      |-5              |Cash on Delivery|Coimbatore|Tamil Nadu    |2025-08-19|Delivered   |
|100102  |C0032      |Anjali Kumar    |Atta 10kg             |Grocery        |7       |0         |40              |Credit Card     |Mumbai    |Maharashtra   |2025-03-02|Processing  |
|100114  |C0252      |Keerthi Singh   |Ceiling F

**Invalid Dates:** Some dates may use different separators or day/month order. We attempt (`yyyy-MM-dd`), slash-separated ISO, day-first slash, and day-first dash formats. Dates that cannot be parsed safely become null and we do not guess ambiguous month-first dates.

In [33]:
def parsed_date(column_name):
    raw = F.trim(F.col(column_name).cast("string"))
    # try_to_timestamp returns null rather than failing on invalid dates in Spark ANSI mode.
    formats = ["yyyy-MM-dd", "yyyy/MM/dd", "dd/MM/yyyy", "dd-MM-yyyy"]
    return F.coalesce(*[
        F.expr(f"try_to_timestamp(`{column_name}`, '{fmt}')").cast("date")
        for fmt in formats
    ])

invalid_dates = df.filter(
    F.col("order_date").isNull() |
    (F.trim(F.col("order_date")) == "") |
    parsed_date("order_date").isNull()
)
print("Invalid/unparseable date rows:", invalid_dates.count())
invalid_dates.select("order_id", "order_date").show(20, truncate=False)

Invalid/unparseable date rows: 5
+--------+----------+
|order_id|order_date|
+--------+----------+
|100097  |31/02/2026|
|100448  |not-a-date|
|100634  |2025-02-30|
|100637  |2026-13-05|
|100969  |09-15-2026|
+--------+----------+



#### 3. Basic Preprocessing - Silver Layer
1 Remove exact duplicate rows.

2 Handle missing values using a reasonable method.

3 Trim extra spaces and standardize obvious capitalization inconsistencies in text fields.

4 Apply basic validation to quantity, unit price, and discount percentage; handle clearly invalid values appropriately.

5 Convert valid dates to one consistent format and handle clearly invalid date values.

6 Preserve useful records where possible. Do not try to make the dataset perfectly clean or remove every unusual value. Save the resulting DataFrame inside the Silver folder in CSV or Parquet format.

In [34]:
# 1. remove exact duplicate rows
cleaned = df.dropDuplicates()

In [35]:
# Whitespace Normalization & Blank-to-Null Conversion

columns = [
    "customer_id",
    "customer_name",
    "product",
    "category",
    "payment_method",
    "city",
    "state",
    "order_status"
]

# capitalization & acronym preservation
for name in ["customer_name", "product", "category", "payment_method", "city", "state", "order_status"]:
    cleaned = cleaned.withColumn(name, F.initcap(F.col(name)))


for colum in columns:
  normalized = F.regexp_replace(F.trim(F.col(colum)), r"\s+", " ")
  cleaned = cleaned.withColumn(name, F.when(normalized == "", F.lit(None)).otherwise(normalized))

In [36]:
# impute mimssing text values
cleaned = cleaned.fillna({
    "customer_name": "NaN",
    "payment_method": "NaN",
    "city": "NaN"
})

In [37]:
# nullify invalid numeric values
for name, condition in {
    "quantity": F.col("quantity") > 0,
    "unit_price": F.col("unit_price") > 0,
    "discount_percent": F.col("discount_percent").between(0, 100)
}.items():
    cleaned = cleaned.withColumn(
        name, F.when(condition, F.col(name)).otherwise(F.lit(None))
    )

In [38]:
#  date parsing & verification
cleaned = cleaned.withColumn("order_date", parsed_date("order_date"))
print("Silver preprocessing completed")
cleaned.show(5, truncate=False)


Silver preprocessing completed
+--------+-----------+------------------+----------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|order_id|customer_id|customer_name     |product               |category |quantity|unit_price|discount_percent|payment_method  |city      |state      |order_date|order_status|
+--------+-----------+------------------+----------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports   |2       |670       |30              |Credit Card     |Bengaluru |Karnataka  |2025-06-06|Karnataka   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books    |4       |670       |0               |Net Banking     |Pune      |Maharashtra|2025-05-08|Maharashtra |
|100506  |C0228      |Diya Tripathi     |Competitive Exam Guide|Books    |6       |610   

##### Save the resulting DataFrame inside the Silver folder in CSV or Parquet format.

In [39]:
import glob

cleaned.coalesce(1).write.mode("overwrite").option("header", True).csv(silver_path)
os.makedirs(os.path.dirname(submission_csv), exist_ok=True)
part_file = glob.glob(os.path.join(silver_path, "part-*.csv"))[0]
shutil.copyfile(part_file, submission_csv)
print("Saved Silver dataset at:", submission_csv)


Saved Silver dataset at: /content/submissions/silver/sales_silver.csv


### 4. Verification
• Show the row count before and after preprocessing.

• Read the Silver data back with Spark and display a few rows.

• Briefly state which preprocessing operations were performed.

In [40]:
after_count = cleaned.count()
print("Before preprocessing:", before_count)
print("After preprocessing:", after_count)
print("Exact duplicates removed:", before_count - after_count)

silver_df = spark.read.option("header", True).option("inferSchema", True).csv(silver_path)
print("Silver count read from disk:", silver_df.count())
silver_df.printSchema()
silver_df.show(10, truncate=False)

assert silver_df.count() == after_count
print("Verified: Silver row count matches expected preprocessed count.")

Before preprocessing: 1000
After preprocessing: 990
Exact duplicates removed: 10
Silver count read from disk: 990
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+--------------+
|order_id|customer_id|customer_name     |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |stat

In [41]:
base_path = "/content/data_lake"

print("Data Lake folder structure:")
for root, dirs, filenames in os.walk(base_path):
    level = root.replace(base_path, "").count(os.sep)
    print("  " * level + os.path.basename(root) + "/")
    for filename in sorted(filenames):
        print("  " * (level + 1) + filename)

Data Lake folder structure:
data_lake/
  bronze/
    sales.csv
    .ipynb_checkpoints/
  .ipynb_checkpoints/
  silver/
    .ipynb_checkpoints/
    preprocessed_sales/
      ._SUCCESS.crc
      .part-00000-52ecfd24-af05-4b83-9b61-933e7e868ebf-c000.csv.crc
      _SUCCESS
      part-00000-52ecfd24-af05-4b83-9b61-933e7e868ebf-c000.csv


### 8. Short assignment report

First we copy the sales CSV into the Bronze layer. We use PySpark to inspect the schema, sample rows, row count, duplicates, missing values, incosistent text, invalid numeric values, and unparseable dates.


1. we use df.dropDuplicates() to perform standard deduplication. Then it identify and dropped exactly 10 duplicate rows.


2. Standardizing Text Fields

Trimming and Collapsing Whitespace: We trim extra leading and trailing spaces and multiple spaces inside sentences like " Hyderabad " or "Hyderabad " were stripped or collapsed into a single space using F.trim() and F.regexp_replace() with regular expressions.


Standardizing Case Representation: Varied casing in categorical columns (e.g., CLOTHING, Clothing, beauty, sports) was normalized into uniform Proper/Title Case using F.initcap().


Empty String Conversion: We converted empty text inputs "" or spaces-only columns directly to NULL values in order to treat them  as missing data.


3. Missing Value Imputation


Handling Nulls: Important categorical columns that contain missing values, specifically customer_name, payment_method, and city; were filled with a value "Unknown" using .fillna() in order to preserve useful record


4. Validating and Nullifying Invalid Numeric Fields


Quantity Validation: Filtered out negative or zero items by setting quantities $\le 0$$\le 0$ to NULL.


Unit Price Validation: We find and nullify negative or zero prices (prices $\le 0$ became NULL).


Discount Percentage Validation: We check the discounts fell within a logical $0\%$ to $100\%$ range. All discounts outside range (such as -5% or 120%) were nullified.


5. Standardizing Dates (Handling Calendar Inconsistencies)


Multi-Format Parsing: Leveraged a custom parser running coalesce over different format patterns (e.g., yyyy-MM-dd, yyyy/MM/dd, dd/MM/yyyy, and dd-MM-yyyy).


  Invalid Date Nullification: Successfully We caught logical calendar errors (like 31/02/2026 or 2025-02-30) and completely invalid strings (like "not-a-date").
  

We have so far 1,000 original rows initially then 10 exact duplicates, and 990 rows after deduplication. We may confirm the actual values by using the executed outputs.